<a href="https://colab.research.google.com/github/nikamsudarshan/movie-recommendation-engine/blob/main/movie_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re
import pandas as pd
import numpy as np

def clean_text(text):
    """Applies lowercasing, strips punctuation, and normalizes whitespaces."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_metadata_soup(df):
    """Combines core descriptive text attributes into a single feature column."""
    df = df.copy()
    # Using 'genres' and 'original_title' as standard proxy fields for standard datasets
    df['genres'] = df.get('genres', pd.Series([''] * len(df))).fillna('')
    df['overview'] = df.get('overview', pd.Series([''] * len(df))).fillna('')

    df['metadata_soup'] = df['overview'] + " " + df['genres']
    df['cleaned_soup'] = df['metadata_soup'].apply(clean_text)
    return df

def generate_interaction_matrix(ratings_df):
    """Converts a long ratings dataframe into a sparse pivot table for KNN."""
    interaction_matrix = ratings_df.pivot(
        index='userId',
        columns='movieId',
        values='rating'
    ).fillna(0)
    return interaction_matrix

In [6]:
import pandas as pd
import urllib.request
import zipfile

# 1. Fetch the official MovieLens dataset (Small version) directly from the source
print("Downloading official dataset from GroupLens...")
zip_url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
urllib.request.urlretrieve(zip_url, "ml-latest-small.zip")

# Extract the zip file in the Colab environment
with zipfile.ZipFile("ml-latest-small.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# Load the newly extracted CSVs
movies_df = pd.read_csv("ml-latest-small/movies.csv")
ratings_df = pd.read_csv("ml-latest-small/ratings.csv")

# 2. Process the Data
print("Processing Text Pipeline...")
# MovieLens groups genres with the '|' character (e.g., Action|Adventure|Sci-Fi).
# We will clean that up to build our text soup.
movies_df['overview'] = movies_df['genres'].str.replace('|', ' ')
processed_movies = build_metadata_soup(movies_df)

print("Processing Interaction Matrix...")
interaction_grid = generate_interaction_matrix(ratings_df)

# 3. View the Results
print("\n--- Processed Text Data Sample ---")
print(processed_movies[['title', 'cleaned_soup']].head(3))

print("\n--- Interaction Matrix Structural Shape ---")
print(f"Users: {interaction_grid.shape[0]} | Movies: {interaction_grid.shape[1]}")

Processing Text Pipeline...
Processing Interaction Matrix...

--- Processed Text Data Sample ---
                     title                                       cleaned_soup
0         Toy Story (1995)  adventure animation children comedy fantasy ad...
1           Jumanji (1995)  adventure children fantasy adventurechildrenfa...
2  Grumpier Old Men (1995)                       comedy romance comedyromance

--- Interaction Matrix Structural Shape ---
Users: 610 | Movies: 9724


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Building TF-IDF Matrix...")
# Initialize the vectorizer and remove common English stopwords ('the', 'a', 'in')
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform our cleaned text soup into numerical vectors
tfidf_matrix = tfidf.fit_transform(processed_movies['cleaned_soup'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape} (Movies x Unique Words)")

print("Calculating Cosine Similarity...")
# Calculate the mathematical distance between all 9,724 movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a reverse mapping of movie titles to their index numbers for quick lookups
indices = pd.Series(processed_movies.index, index=processed_movies['title']).drop_duplicates()

def get_content_recommendations(title, cosine_sim_matrix=cosine_sim, df=processed_movies, indices_map=indices, top_n=5):
    """
    Returns the top_n movies that are most contextually similar to the target title.
    """
    if title not in indices_map:
        return f"Error: Movie '{title}' not found in the database."

    # Get the index of the movie
    idx = indices_map[title]

    # Get the pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies (ignoring the 0th index, which is the movie itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top recommended titles and their genres to verify the logic
    return df[['title', 'genres']].iloc[movie_indices]

# Let's test the engine!
test_movie = "Toy Story (1995)"
print(f"\n--- Content-Based Recommendations for: {test_movie} ---")
print(get_content_recommendations(test_movie))

Building TF-IDF Matrix...
TF-IDF Matrix Shape: (9742, 953) (Movies x Unique Words)
Calculating Cosine Similarity...

--- Content-Based Recommendations for: Toy Story (1995) ---
                                               title  \
1706                                     Antz (1998)   
2355                              Toy Story 2 (1999)   
2809  Adventures of Rocky and Bullwinkle, The (2000)   
3000                Emperor's New Groove, The (2000)   
3568                           Monsters, Inc. (2001)   

                                           genres  
1706  Adventure|Animation|Children|Comedy|Fantasy  
2355  Adventure|Animation|Children|Comedy|Fantasy  
2809  Adventure|Animation|Children|Comedy|Fantasy  
3000  Adventure|Animation|Children|Comedy|Fantasy  
3568  Adventure|Animation|Children|Comedy|Fantasy  


In [8]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

print("Preparing Matrix for KNN...")
# For item-based collaborative filtering, we need movies as rows and users as columns.
# Currently, our interaction_grid has users as rows and movies as columns, so we transpose it (.T)
movie_user_matrix = interaction_grid.T

# Convert the grid to a Sparse Matrix.
# This is a crucial ML scaling technique that ignores '0' values to save RAM.
movie_user_matrix_sparse = csr_matrix(movie_user_matrix.values)

print("Training KNN Model...")
# Initialize KNN with the cosine metric to measure the angle between user-rating vectors
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6)
knn.fit(movie_user_matrix_sparse)

def get_collaborative_recommendations(title, df=processed_movies, interaction_df=movie_user_matrix, model=knn, top_n=5):
    """
    Returns the top_n movies based on user rating patterns using KNN.
    """
    # 1. Find the movieId for the given title
    try:
        movie_id = df.loc[df['title'] == title, 'movieId'].values[0]
    except IndexError:
        return f"Error: Movie '{title}' not found."

    # 2. Get the row index of this movie in the transposed interaction matrix
    try:
        movie_idx = interaction_df.index.get_loc(movie_id)
    except KeyError:
         return f"Error: No ratings found for '{title}'."

    # 3. Find the nearest neighbors based on rating vectors
    # We reshape the data to a 2D array as required by sklearn
    distances, indices = model.kneighbors(interaction_df.iloc[movie_idx, :].values.reshape(1, -1), n_neighbors=top_n+1)

    # 4. Extract the recommended movie IDs (ignoring index 0, which is the queried movie itself)
    recommendation_ids = [interaction_df.index[i] for i in indices.flatten()[1:]]

    # 5. Return the titles and genres for verification
    # We use a categorical trick to maintain the exact order returned by KNN
    df_result = df[df['movieId'].isin(recommendation_ids)].copy()
    df_result['movieId'] = pd.Categorical(df_result['movieId'], categories=recommendation_ids, ordered=True)

    return df_result.sort_values('movieId')[['title', 'genres']]

# Let's test the collaborative engine!
test_movie = "Toy Story (1995)"
print(f"\n--- Collaborative Recommendations for: {test_movie} ---")
print(get_collaborative_recommendations(test_movie))

Preparing Matrix for KNN...
Training KNN Model...

--- Collaborative Recommendations for: Toy Story (1995) ---
                                          title  \
2355                         Toy Story 2 (1999)   
418                        Jurassic Park (1993)   
615        Independence Day (a.k.a. ID4) (1996)   
224   Star Wars: Episode IV - A New Hope (1977)   
314                         Forrest Gump (1994)   

                                           genres  
2355  Adventure|Animation|Children|Comedy|Fantasy  
418              Action|Adventure|Sci-Fi|Thriller  
615              Action|Adventure|Sci-Fi|Thriller  
224                       Action|Adventure|Sci-Fi  
314                      Comedy|Drama|Romance|War  


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def get_hybrid_recommendations(title, content_weight=0.5, collab_weight=0.5, top_n=5):
    """
    Merges TF-IDF and KNN scores into a single weighted hybrid recommendation list.
    """
    # 1. Find the target Movie ID
    try:
        movie_id = processed_movies.loc[processed_movies['title'] == title, 'movieId'].values[0]
        # Get index for the TF-IDF matrix
        content_idx = processed_movies.index[processed_movies['title'] == title].tolist()[0]
    except IndexError:
        return f"Error: Movie '{title}' not found."

    # 2. Generate Content Scores (Text Similarity)
    # We grab the array of similarity scores for the target movie against all 9742 movies
    content_scores = cosine_sim[content_idx]
    content_df = pd.DataFrame({'movieId': processed_movies['movieId'], 'content_score': content_scores})

    # 3. Generate Collaborative Scores (Behavioral Similarity)
    try:
        # Get the row index for the interaction grid
        collab_idx = movie_user_matrix.index.get_loc(movie_id)
        # Calculate Cosine Similarity on the sparse rating matrix for the target movie vs all 9724 movies
        collab_sim = cosine_similarity(movie_user_matrix_sparse[collab_idx], movie_user_matrix_sparse).flatten()
        collab_df = pd.DataFrame({'movieId': movie_user_matrix.index, 'collab_score': collab_sim})
    except KeyError:
        # Fallback if a movie exists in text but has absolutely no user ratings
        collab_df = pd.DataFrame({'movieId': movie_user_matrix.index, 'collab_score': 0.0})

    # 4. Merge DataFrames and Calculate Hybrid Score
    # We use a 'left' merge to ensure movies missing from the ratings table don't break the pipeline
    hybrid_df = pd.merge(content_df, collab_df, on='movieId', how='left').fillna(0.0)

    # The Core Algorithm: Weighted Average
    hybrid_df['hybrid_score'] = (hybrid_df['content_score'] * content_weight) + (hybrid_df['collab_score'] * collab_weight)

    # 5. Filter out the target movie itself, sort by the highest score, and slice the top N
    hybrid_df = hybrid_df[hybrid_df['movieId'] != movie_id]
    top_movies = hybrid_df.sort_values(by='hybrid_score', ascending=False).head(top_n)

    # 6. Attach Movie Titles and Genres for human readability
    result = pd.merge(top_movies, processed_movies[['movieId', 'title', 'genres']], on='movieId', how='left')

    # Formatting the output to look like a professional dashboard
    return result[['title', 'genres', 'hybrid_score', 'content_score', 'collab_score']].round(3)

# --- Let's Test the Engine with different tuning weights! ---
test_movie = "Toy Story (1995)"

print(f"--- 50/50 BLEND (Balanced) ---")
print(get_hybrid_recommendations(test_movie, content_weight=0.5, collab_weight=0.5))

print(f"\n--- 80% TEXT / 20% BEHAVIOR (Content-Heavy) ---")
print(get_hybrid_recommendations(test_movie, content_weight=0.8, collab_weight=0.2))

print(f"\n--- 20% TEXT / 80% BEHAVIOR (Collaborative-Heavy) ---")
print(get_hybrid_recommendations(test_movie, content_weight=0.2, collab_weight=0.8))

--- 50/50 BLEND (Balanced) ---
                              title  \
0                Toy Story 2 (1999)   
1             Monsters, Inc. (2001)   
2                       Antz (1998)   
3  Emperor's New Groove, The (2000)   
4            Shrek the Third (2007)   

                                        genres  hybrid_score  content_score  \
0  Adventure|Animation|Children|Comedy|Fantasy         0.786            1.0   
1  Adventure|Animation|Children|Comedy|Fantasy         0.752            1.0   
2  Adventure|Animation|Children|Comedy|Fantasy         0.680            1.0   
3  Adventure|Animation|Children|Comedy|Fantasy         0.639            1.0   
4  Adventure|Animation|Children|Comedy|Fantasy         0.592            1.0   

   collab_score  
0         0.573  
1         0.505  
2         0.361  
3         0.278  
4         0.184  

--- 80% TEXT / 20% BEHAVIOR (Content-Heavy) ---
                              title  \
0                Toy Story 2 (1999)   
1             Monsters, 